<a href="https://colab.research.google.com/github/vaddadisravani201/DubaiLandCoverClassification/blob/main/DubaiLandcoverClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==================================================================
# Dubai Land Cover Classification (Sentinel-2) using Polygons
# Optimized version: raw bands + balanced sampling + ANN + RF
# Classes: 0=Desert, 1=Urban, 2=Water, 3=Vegetation
# ==================================================================

# 1. INSTALL LIBRARIES
!pip install earthengine-api geemap tensorflow scikit-learn

# 2. IMPORT LIBRARIES
import ee
import geemap
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

# 3. AUTHENTICATE & INITIALIZE EE
ee.Authenticate()
ee.Initialize(project='satelliteanalysis')
print("Earth Engine initialized successfully.")

# 4. LOAD DUBAI BOUNDARY
adm1 = ee.FeatureCollection("FAO/GAUL/2015/level1")
dubai = adm1.filter(ee.Filter.And(
    ee.Filter.eq('ADM0_NAME','United Arab Emirates'),
    ee.Filter.eq('ADM1_NAME','Dubai')
))
dubai_geom = dubai.geometry()
print("Dubai boundary loaded.")

# 5. CLOUD MASK FUNCTION
def maskS2clouds(image):
    qa = image.select('QA60')
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11
    mask = qa.bitwiseAnd(cloudBitMask).eq(0).And(
           qa.bitwiseAnd(cirrusBitMask).eq(0))
    return image.updateMask(mask)

# 6. LOAD SENTINEL-2 COMPOSITE (Jan-Jun 2025)
s2 = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
      .filterBounds(dubai_geom)
      .filterDate('2025-01-01', '2025-06-30')
      .map(maskS2clouds)
      .median()
      .clip(dubai_geom))
image = s2.select(['B2','B3','B4','B8'])
print("Sentinel-2 composite ready.")

# 7. DEFINE TRAINING POLYGONS (your original polygons)
# Example: DESERT
desert = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Polygon([[[55.2669,24.7407],[55.2669,24.7311],
                                     [55.2838,24.7311],[55.2838,24.7407]]]), {'class':0}),
    ee.Feature(ee.Geometry.Polygon([[[56.0237,24.9111],[56.0237,24.8527],
                                     [56.2369,24.8527],[56.2369,24.9111]]]), {'class':0}),
    ee.Feature(ee.Geometry.Polygon([[[55.2940,25.3377],[55.2940,25.3317],
                                     [55.3052,25.3317],[55.3052,25.3377]]]), {'class':0})
])

# URBAN
urban = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Polygon([[[55.3915,25.3634],[55.3915,25.3257],
                                     [55.4611,25.3257],[55.4611,25.3634]]]), {'class':1}),
    ee.Feature(ee.Geometry.Polygon([[[55.3816,25.1575],[55.3816,25.1532],
                                     [55.3863,25.1532],[55.3863,25.1575]]]), {'class':1})
])

# WATER
water = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Polygon([[[56.1116,24.7805],[56.1116,24.7794],
                                     [56.1138,24.7794],[56.1138,24.7805]]]), {'class':2}),
    ee.Feature(ee.Geometry.Polygon([[[55.39654007363677, 24.834213898063226],
                                     [55.39668759513259, 24.834194424648246],
                                     [55.3970496933496, 24.83419685882529],
                                     [55.39736351180434, 24.834179819585017],
                                     [55.3976075928247, 24.834194424648246],
                                     [55.39792677569747, 24.834155477809134],
                                     [55.398098437074424, 24.834087320811186],
                                     [55.39827278066039, 24.833943704156972],
                                     [55.39824021272892, 24.833804837725214],
                                     [55.39810073786015, 24.833622273736452],
                                     [55.397561613848154, 24.833649049804976],
                                     [55.397381905844156, 24.833549248065577],
                                     [55.397188786795084, 24.833541945496126],
                                     [55.39698493890995, 24.83348352492497],
                                     [55.39707076959843, 24.833330170794508],
                                     [55.39746773653263, 24.83327661851375],
                                     [55.397770826151316, 24.8334129333647],
                                     [55.39832470231289, 24.833286355293783],
                                     [55.39899793677563, 24.83317438227704],
                                     [55.39906096868748, 24.833309480143313],
                                     [55.39899927788014, 24.833543162591074],
                                     [55.39906263461132, 24.833749787459716],
                                     [55.39847254862804, 24.8342195844286],
                                     [55.39802193751354, 24.834316951453008],
                                     [55.39772957673092, 24.83441188422806],
                                     [55.39749086012859, 24.834392410844256],
                                     [55.39724946131725, 24.83446300184604],
                                     [55.39666205754299, 24.834450830986544],
                                     [55.396511853838156, 24.834394845017382],
                                     [55.396511853838156, 24.834280438827808]]]), {'class':2})

])

# VEGETATION
vegetation = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Polygon([[[55.4408,25.1686],[55.4408,25.1683],
                                     [55.4410,25.1683],[55.4410,25.1686]]]), {'class':3}),
    ee.Feature(ee.Geometry.Polygon([[[55.53185740297698, 25.306232517752285],
                                     [55.53138533419036, 25.305543868715397],
                                     [55.53229996746444, 25.3050201048559],
                                     [55.53245017116927, 25.305194693060585],
                                     [55.5322034079399, 25.305548718370165],
                                     [55.53224095886611, 25.305980336868995]]]), {'class':3}),
    ee.Feature(ee.Geometry.Polygon([[[55.34345007982474, 25.23717874968304],
                                     [55.34348763075095, 25.23705016134824],
                                     [55.34356541481238, 25.2369652444488],
                                     [55.34370757189017, 25.23687790129032],
                                     [55.34393824186545, 25.23684150828911],
                                     [55.3441447719596, 25.236860917891118],
                                     [55.34426010694724, 25.236926425274987],
                                     [55.34442908611518, 25.236979801635776],
                                     [55.34461147632819, 25.23699678501836],
                                     [55.344761680033024, 25.23695553965652],
                                     [55.34481264200431, 25.237018620792515],
                                     [55.3446490272544, 25.23712052101999],
                                     [55.344466637041386, 25.23723940451074],
                                     [55.34423864927512, 25.237392254542307],
                                     [55.34402943697196, 25.237489302081627],
                                     [55.34389800873023, 25.237358287885232],
                                     [55.3437129363082, 25.237304911690664],
                                     [55.343479584123905, 25.237314616455055]]]), {'class':3}),
    ee.Feature(ee.Geometry.Polygon([[[55.359868908252146, 24.860815105225825],
                                     [55.359659695948984, 24.860659351310016],
                                     [55.35949876340809, 24.860673953247957],
                                     [55.359359288539316, 24.860552270379106],
                                     [55.35945048364582, 24.860386781485314],
                                     [55.36000301870289, 24.86031377160882],
                                     [55.36234190496388, 24.860547403061876],
                                     [55.36132802995625, 24.861102275993737],
                                     [55.360888147677805, 24.860917318626434],
                                     [55.3605448249239, 24.86082970714537],
                                     [55.36021223100605, 24.86079076868941]]]), {'class':3}),

    ee.Feature(ee.Geometry.Polygon([[[55.40422728467345, 24.83488572900383],
                                     [55.40499976086974, 24.835338482581385],
                                     [55.40560057568908, 24.836020043984266],
                                     [55.40602972913146, 24.83612227787106],
                                     [55.40681829858184, 24.83625858958878],
                                     [55.40761759686828, 24.83653608083609],
                                     [55.40811648774505, 24.8365798951867],
                                     [55.40804138589263, 24.83616122409144],
                                     [55.407955555204154, 24.835927546585378],
                                     [55.40828278470397, 24.835942151442435],
                                     [55.40821841168761, 24.836132014427307],
                                     [55.40826132703185, 24.83644358382276],
                                     [55.40858319211364, 24.836696733379164],
                                     [55.409146456006766, 24.837071588502234],
                                     [55.407671241048575, 24.836755152434062],
                                     [55.40723135877013, 24.83683791271461],
                                     [55.40656617093444, 24.836463056883957],
                                     [55.40549865174651, 24.836200170299566],
                                     [55.40454914975524, 24.83536282412466]]]), {'class':3})
])

# Merge all polygons
training_polygons = desert.merge(urban).merge(water).merge(vegetation)

# 8. CONVERT POLYGONS TO LABEL IMAGE
empty = ee.Image().byte()
label_image = empty.paint(featureCollection=training_polygons, color='class').rename('class')
image_with_labels = image.addBands(label_image)

# 9. STRATIFIED SAMPLING (balanced)
training = image_with_labels.stratifiedSample(
    numPoints=800*4,
    classBand='class',
    classValues=[0,1,2,3],
    classPoints=[800, 800, 800, 800],
    region=dubai_geom,
    scale=10,
    geometries=False
)
print("Stratified sampling completed.")

# 10. CONVERT TO PANDAS
training_dict = training.getInfo()
rows = [f['properties'] for f in training_dict['features']]
df = pd.DataFrame(rows)
print("Class Distribution:")
print(df['class'].value_counts())

# 11. PREPARE DATA FOR ANN
X = df[['B2','B3','B4','B8']].values / 10000.0
y = df['class'].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# 12. BUILD & TRAIN OPTIMIZED ANN
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(4,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(4, activation='softmax')
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
callback = tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
model.fit(X_train, y_train, epochs=100, validation_split=0.2, callbacks=[callback])

# 13. EVALUATE ANN
y_pred = np.argmax(model.predict(X_test), axis=1)
print("ANN Test Accuracy:", model.evaluate(X_test, y_test, verbose=0)[1])
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

# 14. RANDOM FOREST CLASSIFICATION FOR DUBAI
classifier = ee.Classifier.smileRandomForest(200).train(
    features=training,
    classProperty='class',
    inputProperties=['B2','B3','B4','B8']
)
classified_dubai = image.classify(classifier).clip(dubai_geom)

# 15. VISUALIZE WITH GEEMAP
Map = geemap.Map(center=[25.276987, 55.296249], zoom=10)
rgb_vis = {'bands':['B4','B3','B2'], 'min':0, 'max':3000}
class_vis = {'min':0,'max':3,'palette':['yellow','red','blue','green']}

Map.addLayer(image.clip(dubai_geom), rgb_vis, "Sentinel-2 RGB")
Map.addLayer(classified_dubai, class_vis, "Land Cover Classification")
Map.add_legend(
    title="Land Cover Classes",
    colors=[(255,255,0),(255,0,0),(0,0,255),(0,255,0)],
    labels=["Desert","Urban","Water","Vegetation"]
)
Map.addLayerControl()
Map

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 12.9 MB/s eta 0:00:00
